## 🎯 Learning Objectives
* Understand the core intuition behind the K-Nearest Neighbours (KNN) algorithm for both classification and regression.
* Identify the key steps involved in making a prediction using KNN.
* Implement a KNN model using scikit-learn for a classification task.
* Evaluate the performance of a KNN model and interpret its results.
* Discuss the advantages, disadvantages, and typical use cases of KNN, including its computational trade-offs and modern considerations.


## K-Nearest Neighbours (KNN): Intuition and Use Cases

Welcome to this lesson on K-Nearest Neighbours (KNN), a foundational and remarkably intuitive algorithm in machine learning. KNN is a non-parametric, lazy learning algorithm used for both classification and regression tasks. Its simplicity makes it an excellent starting point for understanding distance-based methods.

### The Core Idea: "Birds of a Feather Flock Together"

Imagine you're trying to figure out what kind of person a new acquaintance is. One common approach is to look at their friends. If most of their friends are artists, you might guess they're an artist too. If most are engineers, you might guess engineer. KNN works on a similar principle: **an object is classified by the majority vote of its neighbours, and its value is determined by the average of its neighbours' values.**

In machine learning terms, this means that to classify a new data point, KNN looks at the `K` data points in the training set that are 'closest' to it. The 'closeness' is determined by a distance metric, most commonly Euclidean distance.

### How KNN Works: A Step-by-Step Guide

Let's break down the process for a classification task:

1.  **Choose `K`**: This is the number of neighbours you want to consider. It's a hyperparameter you need to select. A small `K` makes the model sensitive to noise, while a large `K` can smooth out the decision boundaries but might miss fine-grained patterns.

2.  **Calculate Distances**: For a new, unclassified data point, the algorithm calculates its distance to *every single* data point in the training set. Common distance metrics include:
    *   **Euclidean Distance**: The straight-line distance between two points in Euclidean space. This is the most common choice.
    *   **Manhattan Distance**: The sum of the absolute differences of their Cartesian coordinates (like navigating a city grid).
    *   **Minkowski Distance**: A generalization of both Euclidean and Manhattan distances.

3.  **Find the `K` Nearest Neighbours**: After calculating all distances, the algorithm identifies the `K` data points from the training set that have the smallest distances to the new point.

4.  **Vote (for Classification) or Average (for Regression)**:
    *   **Classification**: The new data point is assigned the class label that is most frequent among its `K` nearest neighbours. It's a majority vote.
    *   **Regression**: The new data point is assigned the average (or weighted average) of the target values of its `K` nearest neighbours.

### Key Characteristics:

*   **Non-parametric**: KNN makes no assumptions about the underlying data distribution. It directly uses the data itself.
*   **Lazy Learner**: Unlike eager learners (like neural networks or decision trees) that build a model during training, KNN defers generalization until a prediction is requested. It simply stores the training data and performs computations only when a new query point arrives.
*   **Sensitive to Feature Scaling**: Since KNN relies on distance calculations, features with larger scales will have a disproportionately larger impact on the distance. Therefore, it's crucial to scale your features (e.g., using standardization or normalization) before applying KNN.
*   **Curse of Dimensionality**: In high-dimensional spaces, the concept of 


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# --- 1. Generate a Synthetic Dataset ---
# We'll create a 2D dataset for easy visualization of decision boundaries.
# This dataset will have 2 features and 2 classes.
print("Generating synthetic dataset...")
X, y = make_classification(
    n_samples=300,        # Total number of samples
    n_features=2,         # Number of features (for 2D visualization)
    n_informative=2,      # Number of informative features
    n_redundant=0,        # Number of redundant features
    n_clusters_per_class=1, # Number of clusters per class
    random_state=42       # For reproducibility
)

print(f"Dataset shape: X={X.shape}, y={y.shape}")

# --- 2. Split Data into Training and Testing Sets ---
# It's crucial to evaluate the model on unseen data.
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # stratify ensures balanced classes in splits
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# --- 3. Feature Scaling ---
# KNN is distance-based, so scaling features is essential.
# StandardScaler transforms data to have a mean of 0 and a standard deviation of 1.
print("Scaling features using StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. Initialize and Train the K-Nearest Neighbours Classifier ---
# Let's choose K=5 for this example.
K = 5
print(f"Initializing KNeighborsClassifier with K={K}...")
knn_classifier = KNeighborsClassifier(n_neighbors=K)

# Train the model. For KNN, 'training' simply means storing the data.
print("Training the KNN model (storing data)...")
knn_classifier.fit(X_train_scaled, y_train)

# --- 5. Make Predictions ---
print("Making predictions on the test set...")
y_pred = knn_classifier.predict(X_test_scaled)

# --- 6. Evaluate Model Performance ---
print("\n--- Model Evaluation ---")

# Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Classification Report (precision, recall, f1-score for each class)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# --- 7. Visualize Decision Boundaries (for 2D data) ---
print("\nVisualizing decision boundaries...")

# Create a meshgrid to plot decision boundaries
x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Predict class for each point in the meshgrid
Z = knn_classifier.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdBu)

# Plot the training points
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap=plt.cm.RdBu, edgecolor='k', s=20, label='Training Data')
# Plot the test points
plt.scatter(X_test_scaled[:, 0], X_test_scaled[:, 1], c=y_test, cmap=plt.cm.RdBu, edgecolor='k', marker='*', s=100, label='Test Data')

plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.title(f"KNN Classification (K={K}) Decision Boundaries")
plt.xlabel('Feature 1 (Scaled)')
plt.ylabel('Feature 2 (Scaled)')
plt.legend()
plt.show()

print("KNN demonstration complete!")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a complete workflow for using K-Nearest Neighbours for a binary classification task. Let's break down the output and discuss the implications.

#### Code Output Interpretation:

1.  **Accuracy Score**: This metric tells us the proportion of correctly classified instances out of the total instances in the test set. An accuracy of, for example, `0.9333` means 93.33% of the test data points were correctly classified by our KNN model.

2.  **Classification Report**: This provides a more detailed breakdown for each class:
    *   **Precision**: Out of all instances predicted as a certain class, how many actually belonged to that class? High precision means fewer false positives.
    *   **Recall**: Out of all actual instances of a certain class, how many were correctly predicted? High recall means fewer false negatives.
    *   **F1-score**: The harmonic mean of precision and recall, offering a balance between the two.
    *   **Support**: The number of actual occurrences of each class in the test set.

3.  **Confusion Matrix**: This table visualizes the performance of a classification algorithm. Each row represents the instances in an actual class, while each column represents the instances in a predicted class.
    *   The diagonal elements show the number of correct predictions.
    *   Off-diagonal elements show misclassifications (false positives and false negatives).

4.  **Decision Boundaries Plot**: For 2D data, this plot is incredibly insightful. It shows how the KNN model divides the feature space into regions, with each region corresponding to a predicted class. The boundaries are often irregular and depend heavily on the distribution of the training data and the chosen `K` value. You'll notice that the boundaries are not linear, reflecting KNN's non-parametric nature.

#### Performance Trade-offs and Considerations:

**Advantages (Pros):**

*   **Simplicity and Intuition**: Easy to understand and implement.
*   **No Training Phase (Lazy Learner)**: The model simply stores the training data. All computation happens during prediction, which can be an advantage if the training data changes frequently.
*   **Non-parametric**: Makes no assumptions about the underlying data distribution, making it flexible for complex, non-linear relationships.
*   **Handles Multi-class Problems Naturally**: Easily extends to more than two classes.
*   **Versatile**: Can be used for both classification and regression.

**Disadvantages (Cons):**

*   **Computationally Expensive at Prediction Time**: For every new data point, KNN must calculate distances to *all* training points. This makes it very slow for large datasets, especially in high-dimensional spaces.
*   **Sensitive to Feature Scaling**: Features with larger ranges can dominate the distance calculation, leading to biased results. Feature scaling (as demonstrated in the code) is crucial.
*   **Sensitive to Outliers and Noisy Data**: A single outlier can significantly influence the classification of its nearest neighbours, especially with small `K` values.
*   **Curse of Dimensionality**: In high-dimensional spaces, the concept of 


### Resources for Further Learning

To deepen your understanding of K-Nearest Neighbours and related concepts, explore the following resources:

*   **Scikit-learn Documentation - `KNeighborsClassifier`**: The official documentation for the KNN implementation in scikit-learn, detailing parameters and methods.
    *   [https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

*   **Scikit-learn Documentation - `KNeighborsRegressor`**: For understanding KNN's application in regression tasks.
    *   [https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)

*   **Wikipedia - K-Nearest Neighbors Algorithm**: A comprehensive overview of the algorithm's theory and history.
    *   [https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm)

*   **Towards Data Science - K-Nearest Neighbors (KNN) Algorithm Explained**: A popular blog post offering intuitive explanations and examples.
    *   [https://towardsdatascience.com/k-nearest-neighbors-knn-algorithm-explained-dc0e21f925e8](https://towardsdatascience.com/k-nearest-neighbors-knn-algorithm-explained-dc0e21f925e8)

*   **Approximate Nearest Neighbor (ANN) Libraries**: For handling large-scale datasets where exact KNN is too slow, explore these modern libraries:
    *   **FAISS (Facebook AI Similarity Search)**: [https://github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)
    *   **Annoy (Approximate Nearest Neighbors Oh Yeah)**: [https://github.com/spotify/annoy](https://github.com/spotify/annoy)
    *   **HNSW (Hierarchical Navigable Small World)**: Often integrated into other libraries, a highly efficient ANN algorithm.

*   **Google AI Studio / TensorFlow Similarity**: While not directly KNN, these platforms offer tools and concepts for similarity search and embeddings, which are modern extensions of the core idea behind KNN.
    *   [https://ai.google.dev/](https://ai.google.dev/)
    *   [https://www.tensorflow.org/similarity](https://www.tensorflow.org/similarity)
